# P-SVM vs CoRe on Kaggle

1. **Add data:** the dataset with this code (made by `kaggle/make_upload_zip.sh`), plus the *output* of earlier versions of this notebook or teammates' notebooks (their `runs` folders are merged so finished runs are skipped).
2. **Settings:** Internet ON. Accelerator **GPU T4 x2** for Phase 2 plans, **None (CPU)** for Phase 1 plans (CPU sessions don't use the GPU quota).
3. **Secrets** (Add-ons → Secrets): `GEMINI_API_KEY` (VLM labels, Phase 2) and `OPENAI_API_KEY` (only for `core` runs).
4. Set `PLAN` below, then **Save Version → Save & Run All** so it runs in the background for up to 12 h and keeps `/kaggle/working`.

See `KAGGLE_README.md` for the order of plans within the 30 GPU-hour budget.

In [ ]:
# ---- what this session runs (see KAGGLE_README.md) ----
PLAN = 'smoke_phase2'   # smoke_phase1 | smoke_phase2 | phase1 | phase1_extra | phase2_peg | phase2_peg_control | phase2_hammer | phase2_button
LABELS = 'vlm'          # 'scripted' = free pipeline test without API calls (not comparable to the paper)
EST_HOURS = 4           # expected hours per run; the launcher switches to measured times once runs finish
PARALLEL = 1            # Phase 2: 1 per session unless check_setup says 2 fit in RAM. Phase 1 on CPU: 2

In [ ]:
import glob, os, shutil
src = os.path.dirname(sorted(glob.glob('/kaggle/input/**/train_kaggle.py', recursive=True))[0])
shutil.copytree(src, '/tmp/CoRe', dirs_exist_ok=True)   # code lives in /tmp so it isn't saved as output
os.makedirs('/kaggle/working/runs', exist_ok=True)
for d in glob.glob('/kaggle/input/**/runs', recursive=True):   # earlier sessions / teammates
    if os.path.isdir(d):
        shutil.copytree(d, '/kaggle/working/runs', dirs_exist_ok=True)
        print('merged earlier results from', d)
os.chdir('/tmp/CoRe')
print('code from', src)

In [ ]:
!bash /tmp/CoRe/kaggle/setup.sh
PY = open('/tmp/psvm_python').read().strip()

In [ ]:
# API settings. Keep the SAME VLM model for every Phase 2 run of the experiment.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    def secret(name):
        try:
            return _secrets.get_secret(name)
        except Exception:
            return None
except ImportError:
    secret = lambda name: None

os.environ['VLM_MODEL'] = 'gemini-3.5-flash-lite'
os.environ['VLM_BASE_URL'] = 'https://generativelanguage.googleapis.com/v1beta/openai/'
if secret('GEMINI_API_KEY'):
    os.environ['VLM_API_KEY'] = secret('GEMINI_API_KEY')
# CoRe's reward-code LLM (only 'core' runs, ~16 calls). Paper: gpt-4.1-mini (OpenAI, paid).
# Without an OpenAI key the free Gemini key is used instead (note this in your report).
if secret('OPENAI_API_KEY'):
    os.environ['LLM_MODEL'] = 'gpt-4.1-mini'
    os.environ['LLM_API_KEY'] = secret('OPENAI_API_KEY')
else:
    os.environ['LLM_MODEL'] = 'gemini-3.8-flash'
    os.environ['LLM_BASE_URL'] = os.environ['VLM_BASE_URL']
    os.environ['LLM_API_KEY'] = os.environ.get('VLM_API_KEY', '')
os.environ.setdefault('VLM_RPM', '10')   # free tier: set to your limit (aistudio.google.com/rate-limit)

In [ ]:
import torch
GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
check = '--render --api' if GPUS and LABELS == 'vlm' else ('--render' if GPUS else '')
!{PY} scripts/check_setup.py {check}

In [ ]:
gpu_arg = '' if not GPUS else ('--gpus 0,1' if PARALLEL > 1 and GPUS > 1 else '--gpus 0')
threads = 2 if not GPUS else 4
!{PY} scripts/launch.py --plan {PLAN} --out /kaggle/working/runs --parallel {PARALLEL} --threads {threads} --labels {LABELS} --est_hours {EST_HOURS} --session_hours 11.3 {gpu_arg}

In [ ]:
!{PY} scripts/summarize.py /kaggle/working/runs --out /kaggle/working/results {'--include_smoke' if PLAN.startswith('smoke') else ''}